In [ ]:
import json
from pathlib import Path
import pydicom
import torch
from collections import Counter, defaultdict
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import torchxrayvision as xrv
import torchvision
import torch.nn.functional as F
import random


print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

In [ ]:
PROJECT_ROOT = Path.cwd().parent
ANNOTATIONS_PATH = PROJECT_ROOT / "Data" / "pneumonia-challenge-annotations-adjudicated-kaggle_2018.json"
IMAGE_ROOT = Path(r"C:\XrayData\mdai_rsna_project_x9N20BZa_images_2018-07-20-153330")

print("Project:", PROJECT_ROOT)
print("Annotations exist:", ANNOTATIONS_PATH.exists())
print("Image dataset exists:", IMAGE_ROOT.exists())

In [ ]:
with open(ANNOTATIONS_PATH, "r", encoding="utf-8") as f:
    annotations_data = json.load(f)

print("Top-level keys:", list(annotations_data.keys()))
print("Datasets:", len(annotations_data["datasets"]))
print("Studies:", len(annotations_data["datasets"][0]["studies"]))
print("Annotations:", len(annotations_data["datasets"][0]["annotations"]))

In [ ]:
# Get the labels belonging to the "Calculated" group
calculated_group = next(
    group for group in annotations_data["labelGroups"]
    if group["name"] == "Calculated"
)

calculated_labels = {
    label["id"]: label["name"]
    for label in calculated_group["labels"]
}

print("Calculated labels:")
for label_id, label_name in calculated_labels.items():
    print(f"- {label_name}: {label_id}")

In [ ]:
calculated_annotations = [
    annotation
    for annotation in annotations_data["datasets"][0]["annotations"]
    if annotation["labelId"] in calculated_labels
]

print("Calculated annotations:", len(calculated_annotations))

In [ ]:
image_labels = defaultdict(set)

for annotation in calculated_annotations:
    image_id = annotation["SOPInstanceUID"]
    label = calculated_labels[annotation["labelId"]]
    image_labels[image_id].add(label)

label_combinations = Counter(
    tuple(sorted(labels))
    for labels in image_labels.values()
)

for combination, count in label_combinations.items():
    print(f"{combination}: {count}")

In [ ]:
dataset = pd.DataFrame(
    [
        {
            "SOPInstanceUID": image_id,
            "label": next(iter(labels)),
        }
        for image_id, labels in image_labels.items()
    ]
)

dataset.head()

In [ ]:
label_mapping = {
    "Normal": 0,
    "No Lung Opacity / Not Normal": 0,
    "Lung Opacity": 1,
}

dataset["target"] = dataset["label"].map(label_mapping)

dataset.head()

In [ ]:
print(dataset["target"].value_counts())
print("\nMissing targets:", dataset["target"].isna().sum())

In [ ]:
dcm_files = list(IMAGE_ROOT.rglob("*.dcm"))

print("DICOM files found:", len(dcm_files))

In [ ]:
#  Map SOPInstanceUID to DICOM file paths

dcm_paths = {
    file.stem: file
    for file in dcm_files
}

dataset["path"] = dataset["SOPInstanceUID"].map(dcm_paths)

print("Images with a DICOM path:", dataset["path"].notna().sum())
print("Missing DICOM paths:", dataset["path"].isna().sum())

In [ ]:
# Select one example from each class
normal_image = dataset[dataset["label"] == "Normal"].iloc[0]
opacity_image = dataset[dataset["label"] == "Lung Opacity"].iloc[0]

def load_dicom_image(path):
    dicom = pydicom.dcmread(path)
    image = dicom.pixel_array

    # Some DICOM images use MONOCHROME1, where lower values are brighter
    if getattr(dicom, "PhotometricInterpretation", "") == "MONOCHROME1":
        image = image.max() - image

    return image

normal = load_dicom_image(normal_image["path"])
opacity = load_dicom_image(opacity_image["path"])

print("Normal shape:", normal.shape)
print("Opacity shape:", opacity.shape)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].imshow(normal, cmap="gray")
axes[0].set_title("Normal")
axes[0].axis("off")

axes[1].imshow(opacity, cmap="gray")
axes[1].set_title("Lung Opacity")
axes[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
def inspect_image(row):
    ds = pydicom.dcmread(row["path"], stop_before_pixels=True)

    print(f"Label: {row['label']}")
    print(f"Photometric Interpretation: {ds.PhotometricInterpretation}")
    print(f"Bits Stored: {ds.BitsStored}")
    print(f"Window Center: {getattr(ds, 'WindowCenter', 'Not present')}")
    print(f"Window Width: {getattr(ds, 'WindowWidth', 'Not present')}")
    print(f"Rescale Slope: {getattr(ds, 'RescaleSlope', 'Not present')}")
    print(f"Rescale Intercept: {getattr(ds, 'RescaleIntercept', 'Not present')}")
    print()

inspect_image(normal_image)
inspect_image(opacity_image)

In [ ]:
def pixel_stats(image, name):
    print(f"{name}:")
    print(f"  Min:    {image.min()}")
    print(f"  Max:    {image.max()}")
    print(f"  Mean:   {image.mean():.2f}")
    print(f"  Median: {np.median(image):.2f}")
    print(f"  P1:     {np.percentile(image, 1):.2f}")
    print(f"  P99:    {np.percentile(image, 99):.2f}")
    print()

pixel_stats(normal, "Normal")
pixel_stats(opacity, "Lung Opacity")

In [ ]:
random.seed(42)

normal_samples = dataset[dataset["label"] == "Normal"].sample(5, random_state=42)
opacity_samples = dataset[dataset["label"] == "Lung Opacity"].sample(5, random_state=42)

samples = pd.concat([normal_samples, opacity_samples])

for _, row in samples.iterrows():
    image = load_dicom_image(row["path"])
    print(
        f"{row['label']:30} "
        f"mean={image.mean():6.2f} "
        f"median={np.median(image):6.2f} "
        f"min={image.min():3} "
        f"max={image.max():3}"
    )

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(15, 6))

for ax, (_, row) in zip(axes.ravel(), samples.iterrows()):
    image = load_dicom_image(row["path"])

    ax.imshow(image, cmap="gray")
    ax.set_title(row["label"])
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
def normalize_image(image):
    low, high = np.percentile(image, (1, 99))
    image = np.clip(image, low, high)
    image = (image - low) / (high - low)
    return image

fig, axes = plt.subplots(2, 5, figsize=(15, 6))

for ax, (_, row) in zip(axes.ravel(), samples.iterrows()):
    image = load_dicom_image(row["path"])
    image = normalize_image(image)

    ax.imshow(image, cmap="gray", vmin=0, vmax=1)
    ax.set_title(row["label"])
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
seg_model = xrv.baseline_models.chestx_det.PSPNet()
seg_model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seg_model = seg_model.to(device)

print("Device:", device)
print("Targets:", seg_model.targets)

In [ ]:
transform_seg = torchvision.transforms.Compose([
    xrv.datasets.XRayResizer(512)
])

seg_image_512 = transform_seg(seg_image)
seg_tensor_512 = torch.from_numpy(seg_image_512).unsqueeze(0).to(device)

print("Input shape:", seg_tensor_512.shape)

with torch.no_grad():
    segmentation = seg_model(seg_tensor_512)

print("Output shape:", segmentation.shape)

In [ ]:
left_lung = segmentation[0, 4].detach().cpu().numpy()
right_lung = segmentation[0, 5].detach().cpu().numpy()

print("Left Lung:")
print("  Min:", left_lung.min())
print("  Max:", left_lung.max())
print("  Mean:", left_lung.mean())

print("\nRight Lung:")
print("  Min:", right_lung.min())
print("  Max:", right_lung.max())
print("  Mean:", right_lung.mean())

left_lung_prob = torch.sigmoid(segmentation[0, 4]).detach().cpu().numpy()
right_lung_prob = torch.sigmoid(segmentation[0, 5]).detach().cpu().numpy()

print("\n\nLeft Lung probability:")
print("  Min:", left_lung_prob.min())
print("  Max:", left_lung_prob.max())
print("  Mean:", left_lung_prob.mean())

print("\nRight Lung probability:")
print("  Min:", right_lung_prob.min())
print("  Max:", right_lung_prob.max())
print("  Mean:", right_lung_prob.mean())

In [ ]:
# Combine the two lung probabilities
lung_probability = torch.maximum(
    torch.from_numpy(left_lung_prob),
    torch.from_numpy(right_lung_prob)
)

# Resize from 512x512 back to the original 1024x1024
lung_probability_1024 = F.interpolate(
    lung_probability.unsqueeze(0).unsqueeze(0),
    size=opacity.shape,
    mode="bilinear",
    align_corners=False
).squeeze().numpy()

# Threshold after resizing
lung_mask = lung_probability_1024 > 0.5

print("Mask shape:", lung_mask.shape)
print("Lung pixels:", lung_mask.sum())
print("Lung percentage:", lung_mask.mean() * 100)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].imshow(opacity, cmap="gray")
axes[0].set_title("Original X-ray")
axes[0].axis("off")

axes[1].imshow(lung_mask, cmap="gray")
axes[1].set_title("Lung Mask")
axes[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
lung_only = opacity.copy()
lung_only[~lung_mask] = 0

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(opacity, cmap="gray")
axes[0].set_title("Original")
axes[0].axis("off")

axes[1].imshow(lung_mask, cmap="gray")
axes[1].set_title("Lung Mask")
axes[1].axis("off")

axes[2].imshow(lung_only, cmap="gray")
axes[2].set_title("Lung Only")
axes[2].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Find the bounding box containing both lungs
ys, xs = np.where(lung_mask)

y_min, y_max = ys.min(), ys.max()
x_min, x_max = xs.min(), xs.max()

# Add a small padding
padding = 20

y_min = max(0, y_min - padding)
y_max = min(opacity.shape[0], y_max + padding)
x_min = max(0, x_min - padding)
x_max = min(opacity.shape[1], x_max + padding)

lung_cropped = opacity[y_min:y_max, x_min:x_max]

print("Crop coordinates:", (x_min, y_min, x_max, y_max))
print("Cropped shape:", lung_cropped.shape)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(opacity, cmap="gray")
axes[0].set_title("Original")
axes[0].axis("off")

axes[1].imshow(lung_only, cmap="gray")
axes[1].set_title("Lung Only")
axes[1].axis("off")

axes[2].imshow(lung_cropped, cmap="gray")
axes[2].set_title("Lung Cropped")
axes[2].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
def segment_lungs(image):
    # TorchXRayVision normalization
    seg_image = xrv.utils.normalize(
        image,
        maxval=255,
        reshape=True
    )

    # Resize explicitly to the PSPNet's native resolution
    seg_image = torchvision.transforms.Compose([
        xrv.datasets.XRayResizer(512)
    ])(seg_image)

    seg_tensor = torch.from_numpy(seg_image).unsqueeze(0).to(device)

    with torch.no_grad():
        output = seg_model(seg_tensor)

    # Lung channels: 4 = Left Lung, 5 = Right Lung
    left_prob = torch.sigmoid(output[0, 4])
    right_prob = torch.sigmoid(output[0, 5])

    lung_probability = torch.maximum(left_prob, right_prob)

    # Resize probability map back to original resolution
    lung_probability = F.interpolate(
        lung_probability.unsqueeze(0).unsqueeze(0),
        size=image.shape,
        mode="bilinear",
        align_corners=False
    ).squeeze().cpu().numpy()

    lung_mask = lung_probability > 0.5

    # Keep only the lung regions
    lung_only = image.copy()
    lung_only[~lung_mask] = 0

    return lung_only


# Select 12 random images from both classes
inspection_samples = dataset.sample(12, random_state=123)

fig, axes = plt.subplots(3, 4, figsize=(12, 10))

for ax, (_, row) in zip(axes.ravel(), inspection_samples.iterrows()):
    image = load_dicom_image(row["path"])
    lung_only_image = segment_lungs(image)

    ax.imshow(lung_only_image, cmap="gray")
    ax.set_title(row["label"])
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Get the 6th image from the inspection sample
failure_row = inspection_samples.iloc[5]

failure_image = load_dicom_image(failure_row["path"])
failure_lung_only = segment_lungs(failure_image)

# Recreate its mask for visualization
seg_image = xrv.utils.normalize(failure_image, maxval=255, reshape=True)
seg_image = torchvision.transforms.Compose([
    xrv.datasets.XRayResizer(512)
])(seg_image)

seg_tensor = torch.from_numpy(seg_image).unsqueeze(0).to(device)

with torch.no_grad():
    output = seg_model(seg_tensor)

left_prob = torch.sigmoid(output[0, 4])
right_prob = torch.sigmoid(output[0, 5])

lung_probability = torch.maximum(left_prob, right_prob)

lung_probability = F.interpolate(
    lung_probability.unsqueeze(0).unsqueeze(0),
    size=failure_image.shape,
    mode="bilinear",
    align_corners=False
).squeeze().cpu().numpy()

failure_mask = lung_probability > 0.5

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(failure_image, cmap="gray")
axes[0].set_title(f"Original — {failure_row['label']}")
axes[0].axis("off")

axes[1].imshow(failure_mask, cmap="gray")
axes[1].set_title("Lung Mask")
axes[1].axis("off")

axes[2].imshow(failure_lung_only, cmap="gray")
axes[2].set_title("Lung Only")
axes[2].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

axes[0].imshow(failure_image, cmap="gray")
axes[0].set_title("Original")
axes[0].axis("off")

axes[1].imshow(left_prob.cpu().numpy(), cmap="gray", vmin=0, vmax=1)
axes[1].set_title("Left Lung Probability")
axes[1].axis("off")

axes[2].imshow(right_prob.cpu().numpy(), cmap="gray", vmin=0, vmax=1)
axes[2].set_title("Right Lung Probability")
axes[2].axis("off")

axes[3].imshow(failure_image, cmap="gray")
axes[3].imshow(
    failure_mask,
    cmap="jet",
    alpha=0.35
)
axes[3].set_title("Combined Lung Mask")
axes[3].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
artifact_samples = pd.concat([
    dataset[dataset["label"] == "Normal"].sample(20, random_state=2026),
    dataset[dataset["label"] == "Lung Opacity"].sample(20, random_state=2026),
]).reset_index(drop=True)

fig, axes = plt.subplots(5, 8, figsize=(16, 11))

for i, (_, row) in enumerate(artifact_samples.iterrows()):
    image = load_dicom_image(row["path"])
    lung_only_image = segment_lungs(image)

    ax = axes.ravel()[i]
    ax.imshow(lung_only_image, cmap="gray")
    class_name = "N" if row["label"] == "Normal" else "P"
    ax.set_title(f"{class_name}-{i+1}")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
metadata_fields = [
    "Modality",
    "ViewPosition",
    "PatientPosition",
    "BodyPartExamined",
    "StudyDescription",
    "SeriesDescription",
    "Manufacturer",
    "ManufacturerModelName",
    "PhotometricInterpretation",
    "Rows",
    "Columns",
]

def collect_metadata(rows):
    results = {field: [] for field in metadata_fields}

    for _, row in rows.iterrows():
        ds = pydicom.dcmread(row["path"], stop_before_pixels=True)

        for field in metadata_fields:
            value = getattr(ds, field, "MISSING")
            results[field].append(value)

    return results


normal_meta = collect_metadata(
    artifact_samples[artifact_samples["label"] == "Normal"]
)

opacity_meta = collect_metadata(
    artifact_samples[artifact_samples["label"] == "Lung Opacity"]
)

for field in metadata_fields:
    print(f"\n=== {field} ===")
    print("Normal:")
    print(Counter(normal_meta[field]))
    print("Lung Opacity:")
    print(Counter(opacity_meta[field]))

In [ ]:
view_data = []

for _, row in dataset.iterrows():
    ds = pydicom.dcmread(row["path"], stop_before_pixels=True)
    view = getattr(ds, "ViewPosition", "MISSING")

    view_data.append({
        "SOPInstanceUID": row["SOPInstanceUID"],
        "label": row["label"],
        "target": row["target"],
        "ViewPosition": view
    })

view_df = pd.DataFrame(view_data)

print(pd.crosstab(
    view_df["label"],
    view_df["ViewPosition"],
))

print("\nPercentages:")
print(pd.crosstab(
    view_df["label"],
    view_df["ViewPosition"],
    normalize="index"
).mul(100).round(2))